In [ ]:
import pandas as pd
import numpy as np

In [ ]:
PRICE_PROMO_DATA = "/Users/solveigroendalliniger/Desktop/Price_Promo.xlsx"
TRANS_SMOOTH     = 1e-3
J                = 2
N_PROMO          = 4

# Load price/promo data
wb = pd.ExcelFile(PRICE_PROMO_DATA)
df = pd.read_excel(wb, sheet_name=wb.sheet_names[2])
df.columns = df.columns.astype(str).str.strip()

for col in ["WeekNum", "Promo_Brand_1", "Promo_Brand_2"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["WeekNum"]).copy()
df["WeekNum"]      = df["WeekNum"].astype(int)
df["Promo_Brand_1"] = df["Promo_Brand_1"].fillna(0).astype(int)
df["Promo_Brand_2"] = df["Promo_Brand_2"].fillna(0).astype(int)

# Joint promo state index: (p1, p2) → p1 * 2^0 + p2 * 2^1
weekly = df[["WeekNum", "Promo_Brand_1", "Promo_Brand_2"]].drop_duplicates().sort_values("WeekNum").reset_index(drop=True)
weekly["promo_idx"] = (weekly[["Promo_Brand_1", "Promo_Brand_2"]].to_numpy() @ (2 ** np.arange(J))).astype(int)

# Estimate transition matrix with Laplace smoothing
idx    = weekly["promo_idx"].to_numpy()
counts = np.full((N_PROMO, N_PROMO), TRANS_SMOOTH)
np.add.at(counts, (idx[:-1], idx[1:]), 1.0)
PROMO_TRANS = counts / counts.sum(axis=1, keepdims=True)

state_labels = ["$(0,0)$", "$(1,0)$", "$(0,1)$", "$(1,1)$"]
trans_df = pd.DataFrame(PROMO_TRANS, index=state_labels, columns=state_labels)
print(trans_df.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Build LaTeX table
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"\centering")
lines.append(r"\caption{Estimated promotion transition matrix}")
lines.append(r"\label{tab:promotion_transition_matrix}")
lines.append(r"\begin{tabular}{lcccc}")
lines.append(r"\toprule")
lines.append(r"Current state & $(0,0)$ & $(1,0)$ & $(0,1)$ & $(1,1)$ \\")
lines.append(r"\midrule")

for row_label, row in zip(state_labels, PROMO_TRANS):
    cells = " & ".join(f"{v:.4f}" for v in row)
    lines.append(f"{row_label} & {cells} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\begin{flushleft}")
lines.append(r"\footnotesize \textit{Notes:} Each row reports the probability of moving from the current promotion state to each possible promotion state in the following week.")
lines.append(r"\end{flushleft}")
lines.append(r"\end{table}")

latex = "\n".join(lines)
print(latex)